In [1]:
!pip install scikit-learn pandas numpy joblib matplotlib --quiet

import pandas as pd
import numpy as np
import gc
import joblib
import warnings
import matplotlib.pyplot as plt
import os

warnings.filterwarnings("ignore")

from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_val_score, learning_curve
)
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

print("Imports successful")

Imports successful


In [2]:
CSV_PATH = "/kaggle/input/datasets/pes2ug23cs197/master-dataset-csv/master_dataset.csv"

print("Loading dataset...")
master_df = pd.read_csv(CSV_PATH)

label_col    = "disease"
feature_cols = [c for c in master_df.columns if c != label_col]

y_raw = master_df[label_col].values
X_raw = master_df[feature_cols].values.astype(np.float32)

del master_df
gc.collect()

print(f"Full dataset — X: {X_raw.shape} | Classes: {len(np.unique(y_raw))}")

Loading dataset...
Full dataset — X: (190622, 1756) | Classes: 1117


In [3]:
le = LabelEncoder()
y  = le.fit_transform(y_raw)
del y_raw
gc.collect()

joblib.dump(le,           "label_encoder.pkl")
joblib.dump(feature_cols, "feature_cols.pkl")

print(f"{len(le.classes_)} disease classes encoded")
print("label_encoder.pkl and feature_cols.pkl saved")

1117 disease classes encoded
label_encoder.pkl and feature_cols.pkl saved


In [4]:
# Rare-class filter 
MIN_SAMPLES = 30  
counts        = pd.Series(y).value_counts()
valid_classes = counts[counts >= MIN_SAMPLES].index
mask          = np.isin(y, valid_classes)
X_raw, y      = X_raw[mask], y[mask]
print(f"After rare-class filter: {len(y)} rows | {len(np.unique(y))} classes")

After rare-class filter: 187362 rows | 483 classes


In [5]:
# Subsample 
num_classes    = len(np.unique(y))
TARGET_SAMPLES = max(50000, num_classes * 50)  # at least 50 samples per class
print(f"Target samples: {TARGET_SAMPLES}")

if len(y) > TARGET_SAMPLES:
    keep_fraction = TARGET_SAMPLES / len(y)
    X_sampled, _, y_sampled, _ = train_test_split(
        X_raw, y,
        train_size   = keep_fraction,
        random_state = 42,
        stratify     = y
    )
    del X_raw, _
    gc.collect()
    print(f"Sampled: {X_sampled.shape[0]} rows | Classes retained: {len(np.unique(y_sampled))}")
else:
    X_sampled, y_sampled = X_raw, y
    del X_raw
    gc.collect()
    print(f"Using all {len(y_sampled)} rows")

Target samples: 50000
Sampled: 49999 rows | Classes retained: 483


In [6]:
# Safety check 
min_count = pd.Series(y_sampled).value_counts().min()
print(f"Least populated class has {min_count} samples")
assert min_count >= 7, f"Too few samples ({min_count}). Increase MIN_SAMPLES or TARGET_SAMPLES."

Least populated class has 8 samples


In [7]:
# Step 1: 70% train, 30% temp 
X_train, X_temp, y_train, y_temp = train_test_split(
    X_sampled, y_sampled,
    test_size    = 0.30,
    random_state = 42,
    stratify     = y_sampled
)

# Step 2: 50% of temp → 15% val, 15% test 
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size    = 0.50,
    random_state = 42,
    stratify     = y_temp
)

del X_sampled, y_sampled, X_temp, y_temp
gc.collect()

total = len(y_train) + len(y_val) + len(y_test)
joblib.dump({
    "X_train": X_train, "X_val": X_val, "X_test": X_test,
    "y_train": y_train, "y_val": y_val, "y_test": y_test
}, "splits.pkl")

print(f"Train:      {X_train.shape[0]} rows  ({X_train.shape[0]/total*100:.1f}%)")
print(f"Validation: {X_val.shape[0]} rows  ({X_val.shape[0]/total*100:.1f}%)")
print(f"Test:       {X_test.shape[0]} rows  ({X_test.shape[0]/total*100:.1f}%)")
print(f"Features:   {X_train.shape[1]}")
print("splits.pkl saved")

Train:      34999 rows  (70.0%)
Validation: 7500 rows  (15.0%)
Test:       7500 rows  (15.0%)
Features:   1756
splits.pkl saved


In [8]:
rf_model = RandomForestClassifier(
    n_estimators     = 500,
    max_depth        = 20,
    min_samples_leaf = 2,
    max_features     = "sqrt",
    class_weight     = "balanced",
    random_state     = 42,
    n_jobs           = -1
)

print("Training Random Forest...")
rf_model.fit(X_train, y_train)

joblib.dump(rf_model, "rf_model.pkl")
print("rf_model.pkl saved")

Training Random Forest...
rf_model.pkl saved


In [9]:
y_val_pred  = rf_model.predict(X_val)
val_acc     = accuracy_score(y_val, y_val_pred)
train_acc   = accuracy_score(y_train, rf_model.predict(X_train))
gap         = train_acc - val_acc

print(f"Train Accuracy : {train_acc*100:.2f}%")
print(f"Val   Accuracy : {val_acc*100:.2f}%")
print(f"Gap            : {gap*100:.2f}%  {'OK' if gap < 0.10 else 'Check regularization'}")

Train Accuracy : 87.85%
Val   Accuracy : 81.81%
Gap            : 6.03%  OK


In [10]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    rf_model, X_train, y_train,
    cv      = skf,
    scoring = "accuracy",
    n_jobs  = -1
)

print(f"CV Scores : {[f'{s:.4f}' for s in cv_scores]}")
print(f"Mean      : {cv_scores.mean()*100:.2f}%")
print(f"Std Dev   : {cv_scores.std()*100:.2f}%  {'Stable' if cv_scores.std() < 0.02 else 'High variance'}")

CV Scores : ['0.8239', '0.8170', '0.8163', '0.8123', '0.8290']
Mean      : 81.97%
Std Dev   : 0.60%  Stable
